In [68]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

import time
import sys
import requests
import logging
import os

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from scipy import stats
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from statsmodels.tsa.api import SimpleExpSmoothing, Holt, ExponentialSmoothing

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.info("Beginning of script execution")

pd.options.display.max_colwidth = 200  # 100 for long width

2025-07-30 15:59:40,786 - INFO - Beginning of script execution


In [ ]:
df = pd.read_excel("data/dummy2.xlsx", sheet_name="noBTM", skiprows=4)
display(df)


,Brc,Agc,P/N,Desc,DN Price,DR/\nNDR,OH,OO,Book,Alloc\nIn,...,Last GRR,Last Return,Date,Category,Bin Loc,Status,Tot\nAll,Tot\nCall,Tot\nDmd,Brc.PN
0,88,81,10081 S,Dummy 81,10.91,DR,42,84,54,8,...,2000-01-01,NaN,NaN,NaN,999FFF,NaN,880,454,129,88.10081 S
1,88,82,10082 A,Dummy 82A,20.92,DR,61,73,68,78,...,2000-01-01,NaN,NaN,NaN,999FFF,NaN,1256,692,164,88.10082 A
2,88,82,10082 B,Dummy 82B,30.93,DR,22,63,43,76,...,2000-01-01,NaN,NaN,NaN,999FFF,NaN,1612,851,312,88.10082 B
3,99,91,10081 S,Dummy 91,10.91,DR,42,84,54,8,...,2000-01-01,NaN,NaN,NaN,999FFF,NaN,1832,454,675,99.10081 S
4,99,92,10082 S,Dummy 92,20.92,DR,61,73,68,78,...,2000-01-01,NaN,NaN,NaN,999FFF,NaN,3160,692,1256,99.10082 S


In [70]:
logging.info("BEGIN Constructing All Branch Data and Combining It to DF")
#SINGLE AGC
# Convert column names to lowercase for case-insensitive operations
df.columns = df.columns.str.lower()

# Extract numerical part of demand column names and sort accordingly
demand_columns = sorted(
    [col for col in df.columns if col.startswith("d-")],
    key=lambda x: int(x.split("-")[1]),  # Sort by numerical value
    reverse=True  # Ensure ascending order from d-16 to d-1
)


# Normalize "P/N" to uppercase to avoid case-sensitive mismatches
df["p/n"] = df["p/n"].str.upper()

# Group by "agc" and "P/N", summing demand across branches
df_all = df.groupby(["agc", "p/n"], as_index=False)[demand_columns].sum()

# Convert summed demand values into a list (in reversed order)
df_all["d"] = df_all[demand_columns].values.tolist()

# Keep only relevant columns
df_all = df_all[["agc", "p/n", "d"]]

# Insert "branch" column with "ALL"
df_all.insert(0, "branch", "ALL")

# Append aggregated data to the original dataframe
df = pd.concat([df, df_all], ignore_index=True)

logging.info(f"All Branch Data Constructed And Merged With DF With Total Data {len(df)}")
display(demand_columns)
display(df_all)

2025-07-30 15:59:40,864 - INFO - BEGIN Constructing All Branch Data and Combining It to DF
2025-07-30 15:59:40,875 - INFO - All Branch Data Constructed And Merged With DF With Total Data 10


['d-16',
 'd-15',
 'd-14',
 'd-13',
 'd-12',
 'd-11',
 'd-10',
 'd-9',
 'd-8',
 'd-7',
 'd-6',
 'd-5',
 'd-4',
 'd-3',
 'd-2',
 'd-1']

,branch,agc,p/n,d
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]"
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]"
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]"
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]"
4,ALL,92,10082 S,"[256, 240, 224, 208, 192, 176, 160, 144, 128, 112, 96, 80, 64, 48, 32, 16]"


In [71]:


# #MULTIPLE AGC df_all ganti ke df_final nanti
# logging.info("BEGIN Constructing All Branch Data and Combining It to DF")

# # Normalize columns
# df.columns = df.columns.str.lower()
# df["p/n"] = df["p/n"].str.upper()

# # Demand columns sorted from D-16 to D-1
# demand_columns = sorted(
#     [col for col in df.columns if col.startswith("d-")],
#     key=lambda x: int(x.split("-")[1]),
#     reverse=True
# )

# # ---------- Step 1: Convert demand columns to list for each row ----------
# df["d"] = df[demand_columns].values.tolist()
# df = df[["brc", "agc", "p/n", "d"]]

# # ---------- Step 2: Aggregate to create full National summary (ALL AGC) ----------
# df_national_all_agc = df.groupby(["p/n"], as_index=False).agg({
#     "d": lambda rows: [sum(x) for x in zip(*rows)]
# })
# df_national_all_agc.insert(0, "brc", "National")
# df_national_all_agc.insert(1, "agc", "ALL AGC")

# # ---------- Step 3: Combine everything ----------
# df_all = pd.concat([df, df_national_all_agc], ignore_index=True)

# # ---------- Step 4: Drop rows with brc == 'National' and agc != 'ALL AGC' ----------
# df_all = df_all[~((df_all["brc"] == "National") & (df_all["agc"] != "ALL AGC"))]

# logging.info(f"Constructed DF with Per Branch and National ALL AGC only — Total: {len(df_all)} rows")
# print(df_all)


In [72]:
# Calculate Forecast
logging.info("BEGIN Forecast Calculation")
# display(df)

2025-07-30 15:59:40,926 - INFO - BEGIN Forecast Calculation


In [73]:
logging.info("BEGIN Mean, Std, UB Calculation, and Construct Clipping Data")

# Get mean and standard deviation of 12 periods before the last one
df_all["d"] = df_all["d"].apply(lambda x: x if isinstance(x, list) else [])  # Ensure d is a list
df_all['mean_12'] = df_all['d'].apply(lambda x: np.mean(x[-13:-1]))  # Use 12 periods before the last one
df_all['std_12'] = df_all['d'].apply(lambda x: np.std(x[-13:-1]))    # Use 12 periods before the last one

# Get upper bound from mean and std
df_all['ub'] = df_all['mean_12'] + 1.5 * df_all['std_12']

# Limit the original df to upper bound (using the 12 periods before the last one)
df_all['clipped_d'] = df_all.apply(lambda row: np.clip(row['d'][-13:-1], 0, row['ub']).tolist(), axis=1)

# Display the updated DataFrame
display(df_all)


2025-07-30 15:59:40,941 - INFO - BEGIN Mean, Std, UB Calculation, and Construct Clipping Data


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]"
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]"
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]"
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]"
4,ALL,92,10082 S,"[256, 240, 224, 208, 192, 176, 160, 144, 128, 112, 96, 80, 64, 48, 32, 16]",120.0,55.232840,202.849261,"[202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]"


In [74]:
logging.info("BEGIN Moving Average Calculation")

# Calculate Simple Moving Average
df_all['clipped_d_15'] = df_all.apply(lambda row: np.clip(row['d'][:15], 0, row['ub']).tolist(), axis=1)

# Function to compute SMA forecasts for D-13 to D-1 using 3-point averages
def sma_forecast(data):
    sma_values = []
    for i in range(13):  # We want 13 forecast points: D-13 to D-1
        window = data[i:i+3]
        forecast = np.mean(window)  # Equal weights
        sma_values.append(forecast)
    return sma_values

# Apply SMA forecasting logic
df_all['ma'] = df_all['clipped_d_15'].apply(sma_forecast)

# Extract the last forecast (for D-1)
df_all['ma_result'] = df_all['ma'].apply(lambda x: x[-1])

display(df_all.tail())

2025-07-30 15:59:40,971 - INFO - BEGIN Moving Average Calculation


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,ma_result
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",3.0
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",6.0
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",12.0
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 101.42463035441595, 101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 99.6164202362773, 95.14154345147199, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0]",24.0
4,ALL,92,10082 S,"[256, 240, 224, 208, 192, 176, 160, 144, 128, 112, 96, 80, 64, 48, 32, 16]",120.0,55.232840,202.849261,"[202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 202.8492607088319, 202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 199.2328404725546, 190.28308690294398, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0]",48.0


In [75]:
import numpy as np
import pandas as pd

logging.info("BEGIN Weighted Moving Average Calculation")


# Function to compute WMA forecasts for D-13 to D-1
def wma_forecast_with_weights(data, weights):
    wma_values = []
    for i in range(13):  # Forecasting D-13 to D-1 using D-16 to D-2
        window = data[i:i+3]
        forecast = np.sum(np.array(window) * weights) / sum(weights)
        wma_values.append(forecast)
    return wma_values

# Define step size
step = 0.05

# Initialize columns to store best weights, forecasts, and WMA results
df_all['wma_best_w1'] = np.nan
df_all['wma_best_w2'] = np.nan
df_all['wma_best_w3'] = np.nan
df_all['wma_result'] = np.nan
df_all['wma_forecast'] = df_all.apply(lambda _: [], axis=1)  # Initialize as empty lists

# Optimize weights for each row
for idx, row in df_all.iterrows():
    best_rmse = float('inf')
    best_weights = (0.15, 0.25, 0.6)  # Initial weight assumption
    best_forecast = None
    best_full_forecast = None  # Store full forecast array

    # Iterate over valid w1 values
    for w1 in np.round(np.arange(0.15, 0.81, step), 2):  # w1 ≥ 0.15
        for w2 in np.round(np.arange(0.25, 0.86 - w1, step), 2):  # w2 ≥ 0.25 and w1 + w2 ≤ 0.85
            w3 = 1 - (w1 + w2)  # Ensure sum is exactly 1

            # Ensure w3 > w2 > w1
            if w3 > w2 > w1:
                weights = (w1, w2, w3)

                # Compute WMA forecast for this row
                wma_forecast = wma_forecast_with_weights(row['clipped_d_15'], weights)

                # Extract the D-1 prediction (last forecast)
                wma_result = wma_forecast[-1]

                # Extract actual last value of 'd' (D-1)
                d_last = row['d'][-1]

                # Compute RMSE for this row
                rmse = np.sqrt((d_last - wma_result) ** 2)

                # Store best weights if RMSE improves
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_weights = weights
                    best_forecast = wma_result
                    best_full_forecast = wma_forecast  # Store full forecast

    # Store the best weights and forecast for this row
    df_all.at[idx, 'wma_best_w1'] = best_weights[0]
    df_all.at[idx, 'wma_best_w2'] = best_weights[1]
    df_all.at[idx, 'wma_best_w3'] = best_weights[2]
    df_all.at[idx, 'wma_result'] = best_forecast
    df_all.at[idx, 'wma_forecast'] = best_full_forecast  # Store full WMA forecast
    
display(df_all.tail())


2025-07-30 15:59:41,008 - INFO - BEGIN Weighted Moving Average Calculation


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,ma_result,wma_best_w1,wma_best_w2,wma_best_w3,wma_result,wma_forecast
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",3.0,0.15,0.25,0.6,2.55,"[12.678078794301994, 12.678078794301994, 12.271231517720796, 11.5017118191453, 10.55, 9.55, 8.55, 7.55, 6.55, 5.55, 4.55, 3.55, 2.55]"
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",6.0,0.15,0.25,0.6,5.10,"[25.35615758860399, 25.35615758860399, 24.542463035441592, 23.0034236382906, 21.1, 19.1, 17.1, 15.1, 13.1, 11.1, 9.1, 7.1, 5.1]"
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",12.0,0.15,0.25,0.6,10.20,"[50.71231517720798, 50.71231517720798, 49.084926070883185, 46.0068472765812, 42.2, 38.2, 34.2, 30.2, 26.2, 22.2, 18.2, 14.2, 10.2]"
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 101.42463035441595, 101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 99.6164202362773, 95.14154345147199, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0]",24.0,0.15,0.25,0.6,20.40,"[101.42463035441595, 101.42463035441595, 98.16985214176637, 92.0136945531624, 84.4, 76.4, 68.4, 60.4, 52.4, 44.4, 36.4, 28.4, 20.4]"
4,ALL,92,10082 S,"[256, 240, 224, 208, 192, 176, 160, 144, 128, 112, 96, 80, 64, 48, 32, 16]",120.0,55.232840,202.849261,"[202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 202.8492607088319, 202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 199.2328404725546, 190.28308690294398, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0]",48.0,0.15,0.25,0.6,40.80,"[202.8492607088319, 202.8492607088319, 196.33970428353274, 184.0273891063248, 168.8, 152.8, 136.8, 120.8, 104.8, 88.8, 72.8, 56.8, 40.8]"


In [76]:
alpha_ewma = 0.4
def custom_exponential_weighted_moving_average(values, alpha=alpha_ewma):
    ewma_values = [values[0]]  # Start with the first value

    # Apply EWMA formula up to D-2 (i.e., index 11 if length = 12)
    for t in range(1, len(values)):
        if np.isnan(values[t]):
            ewma_t = alpha * 0 + (1 - alpha) * ewma_values[-1]
        else:
            ewma_t = alpha * values[t] + (1 - alpha) * ewma_values[-1]
        ewma_values.append(ewma_t)

    return ewma_values  # This gives you EWMA from D-13 to D-2


def ewma_forecast(data, alpha=alpha_ewma):
    # Calculate EWMA up to D-2
    ewma_up_to_d2 = custom_exponential_weighted_moving_average(data, alpha)

    # Forecast D-1 as same as EWMA at D-2
    ewma_d1 = ewma_up_to_d2[-1]

    # Append D-1 forecast to the EWMA list
    ewma_with_d1 = ewma_up_to_d2 + [ewma_d1]

    # Return full EWMA list (D-13 to D-1) and D-1 forecast
    return ewma_with_d1, ewma_d1

df_all['ewma'], df_all['ewma_result'] = zip(*df_all['clipped_d'].apply(lambda x: ewma_forecast(x[-12:], alpha_ewma)))
display(df_all.tail())


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,ma_result,wma_best_w1,wma_best_w2,wma_best_w3,wma_result,wma_forecast,ewma,ewma_result
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",3.0,0.15,0.25,0.6,2.55,"[12.678078794301994, 12.678078794301994, 12.271231517720796, 11.5017118191453, 10.55, 9.55, 8.55, 7.55, 6.55, 5.55, 4.55, 3.55, 2.55]","[12.678078794301994, 12.406847276581196, 11.844108365948717, 11.10646501956923, 10.263879011741539, 9.358327407044923, 8.414996444226954, 7.448997866536173, 6.469398719921704, 5.481639231953022, 4...",3.493390
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",6.0,0.15,0.25,0.6,5.10,"[25.35615758860399, 25.35615758860399, 24.542463035441592, 23.0034236382906, 21.1, 19.1, 17.1, 15.1, 13.1, 11.1, 9.1, 7.1, 5.1]","[25.35615758860399, 24.813694553162392, 23.688216731897434, 22.21293003913846, 20.527758023483077, 18.716654814089846, 16.82999288845391, 14.897995733072346, 12.938797439843407, 10.963278463906043...",6.986780
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",12.0,0.15,0.25,0.6,10.20,"[50.71231517720798, 50.71231517720798, 49.084926070883185, 46.0068472765812, 42.2, 38.2, 34.2, 30.2, 26.2, 22.2, 18.2, 14.2, 10.2]","[50.71231517720798, 49.627389106324785, 47.37643346379487, 44.42586007827692, 41.055516046966154, 37.43330962817969, 33.65998577690782, 29.79599146614469, 25.877594879686814, 21.926556927812086, 1...",13.973560
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 101.42463035441595, 101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 99.6164202362773, 95.14154345147199, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0]",24.0,0.15,0.25,0.6,20.40,"[101.42463035441595, 101.42463035441595, 98.16985214176637, 92.0136945531624, 84.4, 76.4, 68.4, 60.4, 52.4, 44.4, 36.4, 28.4, 20.4]","[101.42463035441595, 99.25477821264957, 94.75286692758974, 88.85172015655384, 82.11103209393231, 74.86661925635939, 67.31997155381563, 59.59198293228938, 51.75518975937363, 43.85311385562417, 35.9...",27.947121
4,ALL,92,10082 S,"[256, 240, 224, 208, 192, 176, 160, 144, 128, 112, 96, 80, 64, 48, 32, 16]",120.0,55.232840,202.849261,"[202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 202.8492607088319, 202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 199.2328404725546, 190.28308690294398, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0]",

In [77]:
logging.info("BEGIN Linear Reggression Calculation")

#LINEAR REGRESSION
#  Calculate Linear Regression
def lr(x):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    model =  LinearRegression()
    model.fit(df_all[['x']], df_all['y'])
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    return model.predict(df_all[['x']])

df_all['lr'] = df_all['clipped_d'].apply(lambda x: lr(x).tolist())
df_all['lr_result'] = df_all['lr'].apply(lambda x: x[-1:])
display(df_all.tail())

2025-07-30 15:59:41,092 - INFO - BEGIN Linear Reggression Calculation


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,ma_result,wma_best_w1,wma_best_w2,wma_best_w3,wma_result,wma_forecast,ewma,ewma_result,lr,lr_result
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",3.0,0.15,0.25,0.6,2.55,"[12.678078794301994, 12.678078794301994, 12.271231517720796, 11.5017118191453, 10.55, 9.55, 8.55, 7.55, 6.55, 5.55, 4.55, 3.55, 2.55]","[12.678078794301994, 12.406847276581196, 11.844108365948717, 11.10646501956923, 10.263879011741539, 9.358327407044923, 8.414996444226954, 7.448997866536173, 6.469398719921704, 5.481639231953022, 4...",3.493390,"[12.905074516268535, 11.917456101103074, 10.929837685937613, 9.942219270772153, 8.954600855606692, 7.96698244044123, 6.979364025275769, 5.991745610110309, 5.004127194944848, 4.016508779779388, 3.0...",[1.0536535342830042]
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",6.0,0.15,0.25,0.6,5.10,"[25.35615758860399, 25.35615758860399, 24.542463035441592, 23.0034236382906, 21.1, 19.1, 17.1, 15.1, 13.1, 11.1, 9.1, 7.1, 5.1]","[25.35615758860399, 24.813694553162392, 23.688216731897434, 22.21293003913846, 20.527758023483077, 18.716654814089846, 16.82999288845391, 14.897995733072346, 12.938797439843407, 10.963278463906043...",6.986780,"[25.81014903253707, 23.834912202206148, 21.859675371875227, 19.884438541544306, 17.909201711213385, 15.93396488088246, 13.958728050551539, 11.983491220220618, 10.008254389889697, 8.033017559558775...",[2.1073070685660085]
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",12.0,0.15,0.25,0.6,10.20,"[50.71231517720798, 50.71231517720798, 49.084926070883185, 46.0068472765812, 42.2, 38.2, 34.2, 30.2, 26.2, 22.2, 18.2, 14.2, 10.2]","[50.71231517720798, 49.627389106324785, 47.37643346379487, 44.42586007827692, 41.055516046966154, 37.43330962817969, 33.65998577690782, 29.79599146614469, 25.877594879686814, 21.926556927812086, 1...",13.973560,"[51.62029806507414, 47.669824404412296, 43.719350743750454, 39.76887708308861, 35.81840342242677, 31.86792976176492, 27.917456101103078, 23.966982440441235, 20.016508779779393, 16.06603511911755, ...",[4.214614137132017]
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 101.42463035441595, 101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 99.6164202362773, 95.14154345147199, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0]",24.0,0.15,0.25,0.6,20.40,"[101.42463035441595, 101.42463035441595, 98.16985214176637, 92.0136945531624, 84.4, 76.4, 68.4, 60.4, 52.4, 44.4, 36.4, 28.4, 20.4]","[101.42463035441595, 99.25477821264

In [78]:
logging.info("BEGIN Polynomial Reggression Calculation")

#POLYNOMIAL 2ND AND 3RD
# Calculate Polynomial Regression
def pr(x, pr_degree):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)

    X = df_all[['x']]  # Independent variable (reshape to 2D array)
    y = df_all['y']    # Dependent variable

    poly = PolynomialFeatures(degree=pr_degree)  # Create polynomial features
    X_poly = poly.fit_transform(X)  # Transform input features
    poly_model = LinearRegression()  # Initialize linear regression model
    poly_model.fit(X_poly, y)  # Fit polynomial model

    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    X_all_poly = poly.transform(df_all[['x']])
    return poly_model.predict(X_all_poly)  

df_all['pr2'] = df_all['clipped_d'].apply(lambda x: pr(x, 2).tolist())
df_all['pr2_result'] = df_all['pr2'].apply(lambda x: x[-1:])
df_all['pr3'] = df_all['clipped_d'].apply(lambda x: pr(x, 3).tolist())
df_all['pr3_result'] = df_all['pr3'].apply(lambda x: x[-1:])
display(df_all.tail())


2025-07-30 15:59:41,156 - INFO - BEGIN Polynomial Reggression Calculation


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,wma_result,wma_forecast,ewma,ewma_result,lr,lr_result,pr2,pr2_result,pr3,pr3_result
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,2.55,"[12.678078794301994, 12.678078794301994, 12.271231517720796, 11.5017118191453, 10.55, 9.55, 8.55, 7.55, 6.55, 5.55, 4.55, 3.55, 2.55]","[12.678078794301994, 12.406847276581196, 11.844108365948717, 11.10646501956923, 10.263879011741539, 9.358327407044923, 8.414996444226954, 7.448997866536173, 6.469398719921704, 5.481639231953022, 4...",3.493390,"[12.905074516268535, 11.917456101103074, 10.929837685937613, 9.942219270772153, 8.954600855606692, 7.96698244044123, 6.979364025275769, 5.991745610110309, 5.004127194944848, 4.016508779779388, 3.0...",[1.0536535342830042],"[12.824004615566205, 11.88060614623838, 10.92836368774303, 9.96727724008015, 8.997346803249744, 8.01857237725181, 7.030953962086348, 6.034491557753358, 5.029185164252841, 4.015034781584795, 2.9920...",[0.9195196985754972],"[12.755905898976167, 11.886796938655609, 10.971699234663891, 10.018867176890712, 9.036555155225766, 8.03301755955875, 7.016508779779365, 5.995283205777302, 4.97759522744226, 3.9716992346639337, 2....",[1.1073070685662216]
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,5.10,"[25.35615758860399, 25.35615758860399, 24.542463035441592, 23.0034236382906, 21.1, 19.1, 17.1, 15.1, 13.1, 11.1, 9.1, 7.1, 5.1]","[25.35615758860399, 24.813694553162392, 23.688216731897434, 22.21293003913846, 20.527758023483077, 18.716654814089846, 16.82999288845391, 14.897995733072346, 12.938797439843407, 10.963278463906043...",6.986780,"[25.81014903253707, 23.834912202206148, 21.859675371875227, 19.884438541544306, 17.909201711213385, 15.93396488088246, 13.958728050551539, 11.983491220220618, 10.008254389889697, 8.033017559558775...",[2.1073070685660085],"[25.64800923113241, 23.76121229247676, 21.85672737548606, 19.9345544801603, 17.99469360649949, 16.03714475450362, 14.061907924172695, 12.068983115506716, 10.058370328505681, 8.03006956316959, 5.98...",[1.8390393971509944],"[25.511811797952333, 23.773593877311217, 21.943398469327782, 20.037734353781424, 18.073110310451533, 16.0660351191175, 14.03301755955873, 11.990566411554603, 9.95519045488452, 7.943398469327867, 5...",[2.2146141371324433]
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,10.20,"[50.71231517720798, 50.71231517720798, 49.084926070883185, 46.0068472765812, 42.2, 38.2, 34.2, 30.2, 26.2, 22.2, 18.2, 14.2, 10.2]","[50.71231517720798, 49.627389106324785, 47.37643346379487, 44.42586007827692, 41.055516046966154, 37.43330962817969, 33.65998577690782, 29.79599146614469, 25.877594879686814, 21.926556927812086, 1...",13.973560,"[51.62029806507414, 47.669824404412296, 43.719350743750454, 39.7

In [79]:
logging.info("BEGIN Simple Exponential Smoothing Calculation")

alpha_ses = 0.8  # ubah nilai alpha (semakin besar semakin berat ke data terbaru)

#SES
def ses(x, alpha = alpha_ses):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1

    new_data = SimpleExpSmoothing(df_all['y']).fit(smoothing_level=alpha, optimized=False).fittedvalues
    return new_data.tolist()

df_all['ses'] = df_all['clipped_d'].apply(lambda x: ses(x, alpha_ses))
df_all['ses_result'] = df_all['ses'].apply(lambda x: x[-1:])
display(df_all)


2025-07-30 15:59:41,249 - INFO - BEGIN Simple Exponential Smoothing Calculation


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,ewma,ewma_result,lr,lr_result,pr2,pr2_result,pr3,pr3_result,ses,ses_result
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,"[12.678078794301994, 12.406847276581196, 11.844108365948717, 11.10646501956923, 10.263879011741539, 9.358327407044923, 8.414996444226954, 7.448997866536173, 6.469398719921704, 5.481639231953022, 4...",3.493390,"[12.905074516268535, 11.917456101103074, 10.929837685937613, 9.942219270772153, 8.954600855606692, 7.96698244044123, 6.979364025275769, 5.991745610110309, 5.004127194944848, 4.016508779779388, 3.0...",[1.0536535342830042],"[12.824004615566205, 11.88060614623838, 10.92836368774303, 9.96727724008015, 8.997346803249744, 8.01857237725181, 7.030953962086348, 6.034491557753358, 5.029185164252841, 4.015034781584795, 2.9920...",[0.9195196985754972],"[12.755905898976167, 11.886796938655609, 10.971699234663891, 10.018867176890712, 9.036555155225766, 8.03301755955875, 7.016508779779365, 5.995283205777302, 4.97759522744226, 3.9716992346639337, 2....",[1.1073070685662216],"[12.678078794301994, 12.678078794301994, 12.1356157588604, 11.22712315177208, 10.245424630354416, 9.249084926070882, 8.249816985214176, 7.249963397042835, 6.249992679408567, 5.249998535881713, 4.2...",[2.249999988287054]
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,"[25.35615758860399, 24.813694553162392, 23.688216731897434, 22.21293003913846, 20.527758023483077, 18.716654814089846, 16.82999288845391, 14.897995733072346, 12.938797439843407, 10.963278463906043...",6.986780,"[25.81014903253707, 23.834912202206148, 21.859675371875227, 19.884438541544306, 17.909201711213385, 15.93396488088246, 13.958728050551539, 11.983491220220618, 10.008254389889697, 8.033017559558775...",[2.1073070685660085],"[25.64800923113241, 23.76121229247676, 21.85672737548606, 19.9345544801603, 17.99469360649949, 16.03714475450362, 14.061907924172695, 12.068983115506716, 10.058370328505681, 8.03006956316959, 5.98...",[1.8390393971509944],"[25.511811797952333, 23.773593877311217, 21.943398469327782, 20.037734353781424, 18.073110310451533, 16.0660351191175, 14.03301755955873, 11.990566411554603, 9.95519045488452, 7.943398469327867, 5...",[2.2146141371324433],"[25.35615758860399, 25.35615758860399, 24.2712315177208, 22.45424630354416, 20.49084926070883, 18.498169852141764, 16.499633970428352, 14.49992679408567, 12.499985358817135, 10.499997071763426, 8....",[4.499999976574108]
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,"[50.71231517720798, 49.627389106324785, 47.37643346379487, 44.42586007827692, 41.055516046966154, 37.43330962817969, 33.65998577690782, 29.79599146614469, 25.877594879686814, 21.926556927812086, 1...",13.973560,"[51.62029806507414, 47.669824404412296, 43.7

In [80]:
logging.info("BEGIN Double Exponential Smoothing Calculation")

# Define Grid Search Ranges
alpha_values = np.arange(0.1, 1.0, 0.1)  # Alpha range from 0.1 to 0.9
beta_values = np.arange(0.1, 1.0, 0.1)   # Beta range from 0.1 to 0.9

# Double Exponential Smoothing function
def des(x, alpha, beta):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1

    model = ExponentialSmoothing(df_all['y'], trend='add', seasonal=None)
    fitted_model = model.fit(smoothing_level=alpha, smoothing_trend=beta, optimized=False)
    
    return fitted_model.fittedvalues.tolist()

# Function to find the best alpha & beta using Grid Search with RMSE
def optimize_des(series):
    best_alpha, best_beta, best_rmse = None, None, float("inf")

    for alpha, beta in product(alpha_values, beta_values):
        try:
            predictions = des(series, alpha, beta)
            rmse = np.sqrt(mean_squared_error(series, predictions[:len(series)]))  # RMSE calculation

            if rmse < best_rmse:
                best_alpha, best_beta, best_rmse = alpha, beta, rmse

        except Exception as e:
            continue  # Skip if model fails for some values

    return best_alpha, best_beta

# Apply Grid Search Optimization
df_all[['best_alpha', 'best_beta']] = df_all['clipped_d'].apply(lambda x: pd.Series(optimize_des(x)))
df_all['des'] = df_all['clipped_d'].apply(lambda x: des(x, *optimize_des(x)))
df_all['des_result'] = df_all['des'].apply(lambda x: x[-1:])  # Get last predicted value
display(df_all.tail())



2025-07-30 15:59:41,298 - INFO - BEGIN Double Exponential Smoothing Calculation


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,pr2,pr2_result,pr3,pr3_result,ses,ses_result,best_alpha,best_beta,des,des_result
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,"[12.824004615566205, 11.88060614623838, 10.92836368774303, 9.96727724008015, 8.997346803249744, 8.01857237725181, 7.030953962086348, 6.034491557753358, 5.029185164252841, 4.015034781584795, 2.9920...",[0.9195196985754972],"[12.755905898976167, 11.886796938655609, 10.971699234663891, 10.018867176890712, 9.036555155225766, 8.03301755955875, 7.016508779779365, 5.995283205777302, 4.97759522744226, 3.9716992346639337, 2....",[1.1073070685662216],"[12.678078794301994, 12.678078794301994, 12.1356157588604, 11.22712315177208, 10.245424630354416, 9.249084926070882, 8.249816985214176, 7.249963397042835, 6.249992679408567, 5.249998535881713, 4.2...",[2.249999988287054],0.1,0.1,"[12.88879085621341, 11.883171867895774, 10.911475180300743, 9.937833409662208, 8.962177481990905, 7.984465372266823, 7.0046798197924804, 6.022826024367648, 5.038929348241622, 4.053033046245782, 3....",[1.084001137473771]
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,"[25.64800923113241, 23.76121229247676, 21.85672737548606, 19.9345544801603, 17.99469360649949, 16.03714475450362, 14.061907924172695, 12.068983115506716, 10.058370328505681, 8.03006956316959, 5.98...",[1.8390393971509944],"[25.511811797952333, 23.773593877311217, 21.943398469327782, 20.037734353781424, 18.073110310451533, 16.0660351191175, 14.03301755955873, 11.990566411554603, 9.95519045488452, 7.943398469327867, 5...",[2.2146141371324433],"[25.35615758860399, 25.35615758860399, 24.2712315177208, 22.45424630354416, 20.49084926070883, 18.498169852141764, 16.499633970428352, 14.49992679408567, 12.499985358817135, 10.499997071763426, 8....",[4.499999976574108],0.1,0.1,"[25.77758171242682, 23.766343735791548, 21.822950360601485, 19.875666819324415, 17.92435496398181, 15.968930744533646, 14.009359639584961, 12.045652048735295, 10.077858696483244, 8.106066092491565...",[2.168002274947542]
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,"[51.29601846226482, 47.52242458495352, 43.71345475097212, 39.8691089603206, 35.98938721299898, 32.07428950900724, 28.12381584834539, 24.137966231013433, 20.116740657011363, 16.06013912633918, 11.9...",[3.678078794301989],"[51.02362359590467, 47.547187754622435, 43.886796938655564, 40.07546870756285, 36.146220620903065, 32.132070238235, 28.06603511911746, 23.981132823109206, 19.91038090976904, 15.886796938655735, 11...",[4.4292282742648865],"[50.71231517720798, 50.71231517720798, 48.5424630354416, 44.90849260708832, 40.98169852141766, 36.99633970428353, 32.999267940856704, 28.99985358817134, 24.99997071763427, 20.999994143526852, 16.9...",[8.999999953

In [81]:
# Calculate metrics including MASE, MAPE, and SMAPE
def metric(x):
    period_length = len(x['clipped_d'])
    df_all = pd.DataFrame()
    df_all['qty'] = x['clipped_d'][:period_length]  # Ground truth values
    
    # Naive forecast (previous period's value)
    df_all['naive'] = df_all['qty'].shift(1)

    models = ['ma', 'wma_forecast', 'ewma', 'lr', 'pr2', 'pr3', 'ses', 'des']
    for model in models:
        df_all[model] = x[model][:period_length]

    # Compute MASE scaling factor (denominator)
    naive_diff = np.abs(df_all['qty'].diff()).dropna()
    naive_mae = naive_diff.mean() if not naive_diff.empty else np.nan

    result = []
    for model in models:
        y_true = df_all['qty'].dropna()
        y_pred = df_all[model].dropna()
        y_naive = df_all['naive'].dropna()

        # Standard error metrics
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        # Relative errors for MdRAE and GMRAE
        relative_errors = np.abs(y_true - y_pred) / np.abs(y_true - y_naive)
        relative_errors = relative_errors.replace([np.inf, -np.inf], np.nan).dropna()

        # Compute MdRAE and GMRAE
        if not relative_errors.empty:
            mdrae = np.median(relative_errors)
            gmrae = np.exp(np.mean(np.log(relative_errors)))
        else:
            mdrae, gmrae = np.nan, np.nan

        # Compute MASE
        mase = mae / naive_mae if naive_mae > 0 else np.nan

        # Compute MAPE (bounded between 0% - 100%)
        mape_values = np.abs((y_true - y_pred) / y_true)
        mape_values = mape_values.replace([np.inf, -np.inf], np.nan).dropna()
        mape = 100 * mape_values.mean() if not mape_values.empty else np.nan

        # Compute SMAPE (bounded between 0% - 100%)
        smape_values = np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-10)  # Avoid div by zero
        smape_values = smape_values.replace([np.inf, -np.inf], np.nan).dropna()
        smape = 100 * smape_values.mean() if not smape_values.empty else np.nan

        result.append({
            'model': model, 'RMSE': rmse, 'MAE': mae, 'R2': r2,
            'MdRAE': mdrae, 'GMRAE': gmrae, 'MASE': mase, 'MAPE': mape, 'SMAPE': smape
        })

    metrics_df_all = pd.DataFrame(result)

    # Select the best model based on MAE
    best_model_row = metrics_df_all.loc[metrics_df_all['MAE'].idxmin()]
    best_model = best_model_row['model']

    return {'best_model': best_model, 'metrics': metrics_df_all.to_dict(orient='records')}

# Apply metric function
df_all['metric'] = df_all.apply(lambda x: metric(x), axis=1)

# Extract best model and metrics
df_all['best_model'] = df_all['metric'].apply(lambda x: x['best_model'])
df_all['metrics'] = df_all['metric'].apply(lambda x: x['metrics'])
df_all = df_all.drop(columns=['metric'])
# Define the number of months
num_months = 13

# Create new columns dynamically for each month
for i in range(num_months, 0, -1):
    df_all[f'pred_{i}'] = df_all.apply(
        lambda x: x[x['best_model']][num_months - i] if pd.notna(x['best_model']) else np.nan, axis=1
    )

# Extract R² of the best model into a new column
def get_best_model_r2(row):
    best_model = row['best_model']
    for m in row['metrics']:
        if m['model'] == best_model:
            return m['R2']
    return np.nan

df_all['best_r2'] = df_all.apply(get_best_model_r2, axis=1)
# Mark R2 performance
df_all['note'] = np.where(df_all['best_r2'] < 0.25, "R2 < 0.25", "Good")
display(df_all)



,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,pred_8,pred_7,pred_6,pred_5,pred_4,pred_3,pred_2,pred_1,best_r2,note
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,8.033018,7.016509,5.995283,4.977595,3.971699,2.985850,2.028301,1.107307,0.99982,Good
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,16.066035,14.033018,11.990566,9.955190,7.943398,5.971699,4.056602,2.214614,0.99982,Good
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,32.132070,28.066035,23.981133,19.910381,15.886797,11.943398,8.113203,4.429228,0.99982,Good
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 101.42463035441595, 101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 99.6164202362773, 95.14154345147199, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0]",...,64.264140,56.132070,47.962266,39.820762,31.773594,23.886797,16.226406,8.858457,0.99982,Good
4,ALL,92,10082 S,"[256, 240, 224, 208, 192, 176, 160, 144, 128, 112, 96, 80, 64, 48, 32, 16]",120.0,55.232840,202.849261,"[202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 202.8492607088319, 202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 199.2328404725546, 190.28308690294398, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0]",...,128.528281,112.264140,95.924531,79.641524,63.547188,47.773594,32.452812,17.716913,0.99982,Good


In [82]:
#kalkulasi semua model D-0
logging.info("BEGIN Data Selection Calculation")
# Select the best model for each row
df_all['mean_12_FD'] = df_all['d'].apply(lambda x: np.mean(x[-12:]))
df_all['std_12_FD'] = df_all['d'].apply(lambda x: np.std(x[-12:]))
df_all['ub_FD'] = df_all['mean_12_FD'] + 1.5 * df_all['std_12_FD']
df_all['clipped_d_FD'] = df_all.apply(lambda row: np.clip(row['d'][-12:], 0, row['ub_FD']).tolist(), axis=1)
display(df_all)

2025-07-30 15:59:43,561 - INFO - BEGIN Data Selection Calculation


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,pred_4,pred_3,pred_2,pred_1,best_r2,note,mean_12_FD,std_12_FD,ub_FD,clipped_d_FD
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,3.971699,2.985850,2.028301,1.107307,0.99982,Good,6.5,3.452053,11.678079,"[11.678078794301994, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0]"
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,7.943398,5.971699,4.056602,2.214614,0.99982,Good,13.0,6.904105,23.356158,"[23.35615758860399, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0, 2.0]"
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,15.886797,11.943398,8.113203,4.429228,0.99982,Good,26.0,13.808210,46.712315,"[46.71231517720798, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0, 4.0]"
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 101.42463035441595, 101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 99.6164202362773, 95.14154345147199, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0]",...,31.773594,23.886797,16.226406,8.858457,0.99982,Good,52.0,27.616420,93.424630,"[93.42463035441595, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0, 8.0]"
4,ALL,92,10082 S,"[256, 240, 224, 208, 192, 176, 160, 144, 128, 112, 96, 80, 64, 48, 32, 16]",120.0,55.232840,202.849261,"[202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 202.8492607088319, 202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 199.2328404725546, 190.28308690294398, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0]",...,63.547188,47.773594,32.452812,17.716913,0.99982,Good,104.0,55.232840,186.849261,"[186.8492607088319, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0, 16.0]"


In [83]:
logging.info("BEGIN Moving Average Calculation")

# Calculate Simple Moving Average
df_all['clipped_d_15_FD'] = df_all.apply(lambda row: np.clip(row['d'][-15:], 0, row['ub_FD']).tolist(), axis=1)

# Function to compute SMA forecasts for D-13 to D-1 using 3-point averages
def sma_forecast(data):
    sma_values = []
    for i in range(13):  # We want 13 forecast points: D-13 to D-1
        window = data[i:i+3]
        forecast = np.mean(window)  # Equal weights
        sma_values.append(forecast)
    return sma_values

# Apply SMA forecasting logic
df_all['ma_FD'] = df_all['clipped_d_15_FD'].apply(sma_forecast)

# Extract the last forecast (for D-1)
df_all['ma_result_FD'] = df_all['ma_FD'].apply(lambda x: x[-1])

display(df_all)

2025-07-30 15:59:43,609 - INFO - BEGIN Moving Average Calculation


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,pred_1,best_r2,note,mean_12_FD,std_12_FD,ub_FD,clipped_d_FD,clipped_d_15_FD,ma_FD,ma_result_FD
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,1.107307,0.99982,Good,6.5,3.452053,11.678079,"[11.678078794301994, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0]","[11.678078794301994, 11.678078794301994, 11.678078794301994, 11.678078794301994, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0]","[11.678078794301994, 11.678078794301994, 11.452052529534663, 10.892692931433999, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]",2.0
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,2.214614,0.99982,Good,13.0,6.904105,23.356158,"[23.35615758860399, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0, 2.0]","[23.35615758860399, 23.35615758860399, 23.35615758860399, 23.35615758860399, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0, 2.0]","[23.35615758860399, 23.35615758860399, 22.904105059069327, 21.785385862867997, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]",4.0
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,4.429228,0.99982,Good,26.0,13.808210,46.712315,"[46.71231517720798, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0, 4.0]","[46.71231517720798, 46.71231517720798, 46.71231517720798, 46.71231517720798, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0, 4.0]","[46.71231517720798, 46.71231517720798, 45.80821011813865, 43.570771725735995, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]",8.0
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 101.42463035441595, 101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 99.6164202362773, 95.14154345147199, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0]",...,8.858457,0.99982,Good,52.0,27.616420,93.424630,"[93.42463035441595, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0, 8.0]","[93.42463035441595, 93.42463035441595, 93.42463035441595, 93.42463035441595, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0, 8.0]","[93.42463035441595, 93.42463035441595, 91.6164202362773, 87.14154345147199, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]",16.0
4,ALL,92,10082 S,"[256, 240, 224, 208, 192, 176, 160, 144, 128, 112, 96, 80, 64, 48, 32, 16]",120.0,55.232840,202.849261,"[202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 202.8492607088319, 202.8492607088319, 192.0, 176.0, 160.0, 144.0

In [84]:
import numpy as np
import pandas as pd

logging.info("BEGIN Weighted Moving Average Calculation for FD")

# Function to compute WMA forecasts for D-13 to D-1
def wma_forecast_with_weights_FD(data, weights):
    wma_values_FD = []
    for i in range(13):  # Forecasting D-13 to D-1 using D-16 to D-2
        window_FD = data[i:i+3]
        forecast_FD = np.sum(np.array(window_FD) * weights) / sum(weights)
        wma_values_FD.append(forecast_FD)
    return wma_values_FD

# Define step size
step_FD = 0.05

# Initialize columns to store best weights, forecasts, and WMA results
df_all['wma_best_w1_FD'] = np.nan
df_all['wma_best_w2_FD'] = np.nan
df_all['wma_best_w3_FD'] = np.nan
df_all['wma_result_FD'] = np.nan
df_all['wma_forecast_FD'] = df_all.apply(lambda _: [], axis=1)  # Initialize as empty lists

# Optimize weights for each row
for idx, row in df_all.iterrows():
    best_rmse_FD = float('inf')
    best_weights_FD = (0.15, 0.25, 0.6)  # Initial weight assumption
    best_forecast_FD = None
    best_full_forecast_FD = None  # Store full forecast array

    # Iterate over valid w1_FD values
    for w1_FD in np.round(np.arange(0.15, 0.81, step_FD), 2):  # w1_FD ≥ 0.15
        for w2_FD in np.round(np.arange(0.25, 0.86 - w1_FD, step_FD), 2):  # w2_FD ≥ 0.25 and w1_FD + w2_FD ≤ 0.85
            w3_FD = 1 - (w1_FD + w2_FD)  # Ensure sum is exactly 1

            # Ensure w3_FD > w2_FD > w1_FD
            if w3_FD > w2_FD > w1_FD:
                weights_FD = (w1_FD, w2_FD, w3_FD)

                # Compute WMA forecast for this row
                wma_forecast_FD = wma_forecast_with_weights_FD(row['clipped_d_15_FD'], weights_FD)

                # Extract the D-1 prediction (last forecast)
                wma_result_FD = wma_forecast_FD[-1]

                # Extract actual last value of 'd' (D-1)
                d_last_FD = row['d'][-1]

                # Compute RMSE for this row
                rmse_FD = np.sqrt((d_last_FD - wma_result_FD) ** 2)

                # Store best weights if RMSE improves
                if rmse_FD < best_rmse_FD:
                    best_rmse_FD = rmse_FD
                    best_weights_FD = weights_FD
                    best_forecast_FD = wma_result_FD
                    best_full_forecast_FD = wma_forecast_FD  # Store full forecast

    # Store the best weights and forecast for this row
    df_all.at[idx, 'wma_best_w1_FD'] = best_weights_FD[0]
    df_all.at[idx, 'wma_best_w2_FD'] = best_weights_FD[1]
    df_all.at[idx, 'wma_best_w3_FD'] = best_weights_FD[2]
    df_all.at[idx, 'wma_result_FD'] = best_forecast_FD
    df_all.at[idx, 'wma_forecast_FD'] = best_full_forecast_FD  # Store full WMA forecast
    
display(df_all)

2025-07-30 15:59:43,657 - INFO - BEGIN Weighted Moving Average Calculation for FD


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,ub_FD,clipped_d_FD,clipped_d_15_FD,ma_FD,ma_result_FD,wma_best_w1_FD,wma_best_w2_FD,wma_best_w3_FD,wma_result_FD,wma_forecast_FD
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,11.678079,"[11.678078794301994, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0]","[11.678078794301994, 11.678078794301994, 11.678078794301994, 11.678078794301994, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0]","[11.678078794301994, 11.678078794301994, 11.452052529534663, 10.892692931433999, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]",2.0,0.15,0.25,0.6,1.55,"[11.678078794301994, 11.678078794301994, 11.271231517720796, 10.5017118191453, 9.55, 8.55, 7.55, 6.55, 5.55, 4.55, 3.55, 2.55, 1.5499999999999998]"
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,23.356158,"[23.35615758860399, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0, 2.0]","[23.35615758860399, 23.35615758860399, 23.35615758860399, 23.35615758860399, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0, 2.0]","[23.35615758860399, 23.35615758860399, 22.904105059069327, 21.785385862867997, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]",4.0,0.15,0.25,0.6,3.10,"[23.35615758860399, 23.35615758860399, 22.542463035441592, 21.0034236382906, 19.1, 17.1, 15.1, 13.1, 11.1, 9.1, 7.1, 5.1, 3.0999999999999996]"
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,46.712315,"[46.71231517720798, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0, 4.0]","[46.71231517720798, 46.71231517720798, 46.71231517720798, 46.71231517720798, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0, 4.0]","[46.71231517720798, 46.71231517720798, 45.80821011813865, 43.570771725735995, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]",8.0,0.15,0.25,0.6,6.20,"[46.71231517720798, 46.71231517720798, 45.084926070883185, 42.0068472765812, 38.2, 34.2, 30.2, 26.2, 22.2, 18.2, 14.2, 10.2, 6.199999999999999]"
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 101.42463035441595, 101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 99.6164202362773, 95.14154345147199, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0]",...,93.424630,"[93.42463035441595, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0, 8.0]","[93.42463035441595, 93.42463035441595, 93.42463035441595, 93.42463035441595, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0, 8.0]","[93.42463035441595, 93.42463035441595, 91.6164202362773, 87.1415434514

In [85]:
# EWMA
alpha_ewma = 0.4

# Custom Exponential Weighted Moving Average Function
def custom_exponential_weighted_moving_average(values, alpha=alpha_ewma):
    ewma_values = [values[0]]  # Start with the first value (D-12)

    # Apply the EWMA formula for D-11 to D-1 (i.e., 11 more steps)
    for t in range(1, len(values)):  # len(values) = 12
        if np.isnan(values[t]):
            ewma_t = alpha * 0 + (1 - alpha) * ewma_values[-1]
        else:
            ewma_t = alpha * values[t] + (1 - alpha) * ewma_values[-1]
        ewma_values.append(ewma_t)
    
    return ewma_values  # EWMA from D-12 to D-1

# Forecast Function Using the Custom EWMA
def ewma_forecast(data, alpha=alpha_ewma):
    # Compute EWMA values for D-12 to D-1
    ewma_values = custom_exponential_weighted_moving_average(data, alpha)

    # Forecast D-0 as the same as EWMA at D-1
    forecast_d0 = ewma_values[-1]

    # Full series includes D-12 to D-0 (13 values total)
    ewma_full = ewma_values + [forecast_d0]

    return ewma_full, forecast_d0

# Apply the EWMA forecast to the dataset
df_all['ewma_FD'], df_all['ewma_result_FD'] = zip(*df_all['clipped_d_FD'].apply(lambda x: ewma_forecast(x[-12:], alpha_ewma)))

display(df_all)


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,clipped_d_15_FD,ma_FD,ma_result_FD,wma_best_w1_FD,wma_best_w2_FD,wma_best_w3_FD,wma_result_FD,wma_forecast_FD,ewma_FD,ewma_result_FD
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,"[11.678078794301994, 11.678078794301994, 11.678078794301994, 11.678078794301994, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0]","[11.678078794301994, 11.678078794301994, 11.452052529534663, 10.892692931433999, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]",2.0,0.15,0.25,0.6,1.55,"[11.678078794301994, 11.678078794301994, 11.271231517720796, 10.5017118191453, 9.55, 8.55, 7.55, 6.55, 5.55, 4.55, 3.55, 2.55, 1.5499999999999998]","[11.678078794301994, 11.406847276581196, 10.844108365948717, 10.10646501956923, 9.263879011741537, 8.358327407044921, 7.414996444226953, 6.448997866536172, 5.469398719921703, 4.481639231953022, 3....",2.493390
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,"[23.35615758860399, 23.35615758860399, 23.35615758860399, 23.35615758860399, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0, 2.0]","[23.35615758860399, 23.35615758860399, 22.904105059069327, 21.785385862867997, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]",4.0,0.15,0.25,0.6,3.10,"[23.35615758860399, 23.35615758860399, 22.542463035441592, 21.0034236382906, 19.1, 17.1, 15.1, 13.1, 11.1, 9.1, 7.1, 5.1, 3.0999999999999996]","[23.35615758860399, 22.813694553162392, 21.688216731897434, 20.21293003913846, 18.527758023483074, 16.716654814089843, 14.829992888453907, 12.897995733072344, 10.938797439843405, 8.963278463906043...",4.986780
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,"[46.71231517720798, 46.71231517720798, 46.71231517720798, 46.71231517720798, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0, 4.0]","[46.71231517720798, 46.71231517720798, 45.80821011813865, 43.570771725735995, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]",8.0,0.15,0.25,0.6,6.20,"[46.71231517720798, 46.71231517720798, 45.084926070883185, 42.0068472765812, 38.2, 34.2, 30.2, 26.2, 22.2, 18.2, 14.2, 10.2, 6.199999999999999]","[46.71231517720798, 45.627389106324785, 43.37643346379487, 40.42586007827692, 37.05551604696615, 33.433309628179686, 29.659985776907813, 25.795991466144688, 21.87759487968681, 17.926556927812086, ...",9.973560
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 101.42463035441595, 101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 99.6164202362773, 95.14154345147199, 88.0, 80.0, 72

In [86]:
#LR
def lr(x):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    model =  LinearRegression()
    model.fit(df_all[['x']], df_all['y'])
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    return model.predict(df_all[['x']])
df_all['lr_FD'] = df_all['clipped_d_FD'].apply(lambda x: lr(x).tolist())
df_all['lr_result_FD'] = df_all['lr_FD'].apply(lambda x: x[-1:])
display(df_all)

,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,ma_result_FD,wma_best_w1_FD,wma_best_w2_FD,wma_best_w3_FD,wma_result_FD,wma_forecast_FD,ewma_FD,ewma_result_FD,lr_FD,lr_result_FD
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,2.0,0.15,0.25,0.6,1.55,"[11.678078794301994, 11.678078794301994, 11.271231517720796, 10.5017118191453, 9.55, 8.55, 7.55, 6.55, 5.55, 4.55, 3.55, 2.55, 1.5499999999999998]","[11.678078794301994, 11.406847276581196, 10.844108365948717, 10.10646501956923, 9.263879011741537, 8.358327407044921, 7.414996444226953, 6.448997866536172, 5.469398719921703, 4.481639231953022, 3....",2.493390,"[11.905074516268535, 10.917456101103074, 9.929837685937613, 8.942219270772153, 7.954600855606691, 6.96698244044123, 5.979364025275769, 4.991745610110309, 4.004127194944848, 3.0165087797793877, 2.0...",[0.053653534283004234]
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,4.0,0.15,0.25,0.6,3.10,"[23.35615758860399, 23.35615758860399, 22.542463035441592, 21.0034236382906, 19.1, 17.1, 15.1, 13.1, 11.1, 9.1, 7.1, 5.1, 3.0999999999999996]","[23.35615758860399, 22.813694553162392, 21.688216731897434, 20.21293003913846, 18.527758023483074, 16.716654814089843, 14.829992888453907, 12.897995733072344, 10.938797439843405, 8.963278463906043...",4.986780,"[23.81014903253707, 21.834912202206148, 19.859675371875227, 17.884438541544306, 15.909201711213383, 13.93396488088246, 11.958728050551539, 9.983491220220618, 8.008254389889697, 6.033017559558775, ...",[0.10730706856600847]
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,8.0,0.15,0.25,0.6,6.20,"[46.71231517720798, 46.71231517720798, 45.084926070883185, 42.0068472765812, 38.2, 34.2, 30.2, 26.2, 22.2, 18.2, 14.2, 10.2, 6.199999999999999]","[46.71231517720798, 45.627389106324785, 43.37643346379487, 40.42586007827692, 37.05551604696615, 33.433309628179686, 29.659985776907813, 25.795991466144688, 21.87759487968681, 17.926556927812086, ...",9.973560,"[47.62029806507414, 43.669824404412296, 39.719350743750454, 35.76887708308861, 31.818403422426766, 27.86792976176492, 23.917456101103078, 19.966982440441235, 16.016508779779393, 12.06603511911755,...",[0.21461413713201694]
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 101.42463035441595, 101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 99.6164202362773, 95.14154345147199, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0]",...,16.0,0.15,0.25,0.6,12.40,"[93.42463035441595, 93.42463035441595, 90.16985214176637, 84.0136945531624, 7

In [87]:
#PR2&3
def pr(x, pr_degree):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    X = df_all[['x']]  # Independent variable (reshape to 2D array)
    y = df_all['y']    # Dependent variable
    poly = PolynomialFeatures(degree=pr_degree)  # Create polynomial features
    X_poly = poly.fit_transform(X)  # Transform input features
    poly_model = LinearRegression()  # Initialize linear regression model
    poly_model.fit(X_poly, y)  # Fit polynomial model
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    X_all_poly = poly.transform(df_all[['x']])
    return poly_model.predict(X_all_poly)  
df_all['pr2_FD'] = df_all['clipped_d_FD'].apply(lambda x: pr(x, 2).tolist())
df_all['pr2_result_FD'] = df_all['pr2_FD'].apply(lambda x: x[-1:])
df_all['pr3_FD'] = df_all['clipped_d_FD'].apply(lambda x: pr(x, 3).tolist())
df_all['pr3_result_FD'] = df_all['pr3_FD'].apply(lambda x: x[-1:])
display(df_all)

,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,wma_result_FD,wma_forecast_FD,ewma_FD,ewma_result_FD,lr_FD,lr_result_FD,pr2_FD,pr2_result_FD,pr3_FD,pr3_result_FD
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,1.55,"[11.678078794301994, 11.678078794301994, 11.271231517720796, 10.5017118191453, 9.55, 8.55, 7.55, 6.55, 5.55, 4.55, 3.55, 2.55, 1.5499999999999998]","[11.678078794301994, 11.406847276581196, 10.844108365948717, 10.10646501956923, 9.263879011741537, 8.358327407044921, 7.414996444226953, 6.448997866536172, 5.469398719921703, 4.481639231953022, 3....",2.493390,"[11.905074516268535, 10.917456101103074, 9.929837685937613, 8.942219270772153, 7.954600855606691, 6.96698244044123, 5.979364025275769, 4.991745610110309, 4.004127194944848, 3.0165087797793877, 2.0...",[0.053653534283004234],"[11.824004615566205, 10.88060614623838, 9.92836368774303, 8.96727724008015, 7.9973468032497435, 7.018572377251809, 6.030953962086348, 5.034491557753358, 4.029185164252841, 3.015034781584795, 1.992...",[-0.0804803014245028],"[11.755905898976167, 10.886796938655609, 9.971699234663891, 9.018867176890712, 8.036555155225766, 7.033017559558751, 6.016508779779365, 4.995283205777302, 3.9775952274422597, 2.9716992346639337, 1...",[0.10730706856622163]
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,3.10,"[23.35615758860399, 23.35615758860399, 22.542463035441592, 21.0034236382906, 19.1, 17.1, 15.1, 13.1, 11.1, 9.1, 7.1, 5.1, 3.0999999999999996]","[23.35615758860399, 22.813694553162392, 21.688216731897434, 20.21293003913846, 18.527758023483074, 16.716654814089843, 14.829992888453907, 12.897995733072344, 10.938797439843405, 8.963278463906043...",4.986780,"[23.81014903253707, 21.834912202206148, 19.859675371875227, 17.884438541544306, 15.909201711213383, 13.93396488088246, 11.958728050551539, 9.983491220220618, 8.008254389889697, 6.033017559558775, ...",[0.10730706856600847],"[23.64800923113241, 21.76121229247676, 19.85672737548606, 17.9345544801603, 15.994693606499487, 14.037144754503618, 12.061907924172695, 10.068983115506716, 8.058370328505681, 6.03006956316959, 3.9...",[-0.1609606028490056],"[23.511811797952333, 21.773593877311217, 19.943398469327782, 18.037734353781424, 16.073110310451533, 14.066035119117503, 12.03301755955873, 9.990566411554603, 7.955190454884519, 5.943398469327867,...",[0.21461413713244326]
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,6.20,"[46.71231517720798, 46.71231517720798, 45.084926070883185, 42.0068472765812, 38.2, 34.2, 30.2, 26.2, 22.2, 18.2, 14.2, 10.2, 6.199999999999999]","[46.71231517720798, 45.627389106324785, 43.37643346379487, 40.42586007827692, 37.05551604696615, 33.433309628179686, 29.659985776907813, 25.795991466144688, 21.87759487968681, 17.926556927812086, ..."

In [88]:
#SES
def ses(x, alpha = alpha_ses):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    new_data = SimpleExpSmoothing(df_all['y']).fit(smoothing_level=alpha, optimized=False).fittedvalues
    return new_data.tolist()
df_all['ses_FD'] = df_all['clipped_d_FD'].apply(lambda x: ses(x, alpha_ses))
df_all['ses_result_FD'] = df_all['ses_FD'].apply(lambda x: x[-1:])

display(df_all)

,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,ewma_FD,ewma_result_FD,lr_FD,lr_result_FD,pr2_FD,pr2_result_FD,pr3_FD,pr3_result_FD,ses_FD,ses_result_FD
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,"[11.678078794301994, 11.406847276581196, 10.844108365948717, 10.10646501956923, 9.263879011741537, 8.358327407044921, 7.414996444226953, 6.448997866536172, 5.469398719921703, 4.481639231953022, 3....",2.493390,"[11.905074516268535, 10.917456101103074, 9.929837685937613, 8.942219270772153, 7.954600855606691, 6.96698244044123, 5.979364025275769, 4.991745610110309, 4.004127194944848, 3.0165087797793877, 2.0...",[0.053653534283004234],"[11.824004615566205, 10.88060614623838, 9.92836368774303, 8.96727724008015, 7.9973468032497435, 7.018572377251809, 6.030953962086348, 5.034491557753358, 4.029185164252841, 3.015034781584795, 1.992...",[-0.0804803014245028],"[11.755905898976167, 10.886796938655609, 9.971699234663891, 9.018867176890712, 8.036555155225766, 7.033017559558751, 6.016508779779365, 4.995283205777302, 3.9775952274422597, 2.9716992346639337, 1...",[0.10730706856622163],"[11.678078794301994, 11.678078794301992, 11.135615758860398, 10.22712315177208, 9.245424630354416, 8.249084926070884, 7.249816985214177, 6.249963397042836, 5.249992679408567, 4.249998535881714, 3....",[1.2499999882870536]
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,"[23.35615758860399, 22.813694553162392, 21.688216731897434, 20.21293003913846, 18.527758023483074, 16.716654814089843, 14.829992888453907, 12.897995733072344, 10.938797439843405, 8.963278463906043...",4.986780,"[23.81014903253707, 21.834912202206148, 19.859675371875227, 17.884438541544306, 15.909201711213383, 13.93396488088246, 11.958728050551539, 9.983491220220618, 8.008254389889697, 6.033017559558775, ...",[0.10730706856600847],"[23.64800923113241, 21.76121229247676, 19.85672737548606, 17.9345544801603, 15.994693606499487, 14.037144754503618, 12.061907924172695, 10.068983115506716, 8.058370328505681, 6.03006956316959, 3.9...",[-0.1609606028490056],"[23.511811797952333, 21.773593877311217, 19.943398469327782, 18.037734353781424, 16.073110310451533, 14.066035119117503, 12.03301755955873, 9.990566411554603, 7.955190454884519, 5.943398469327867,...",[0.21461413713244326],"[23.35615758860399, 23.356157588603985, 22.271231517720796, 20.45424630354416, 18.49084926070883, 16.498169852141768, 14.499633970428354, 12.499926794085672, 10.499985358817135, 8.499997071763428,...",[2.499999976574107]
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,"[46.71231517720798, 45.627389106324785, 43.37643346379487, 40.42586007827692, 37.05551604696615, 33.433309628179686, 29.659985776907813, 25.795991466144688, 21.87759487968681, 17.926556927812086, ...",9.973560,"[47.620

In [89]:
#DES
# Define Grid Search Ranges
alpha_values_FD = np.arange(0.1, 1.0, 0.1)  # Alpha range from 0.1 to 0.9
beta_values_FD = np.arange(0.1, 1.0, 0.1)   # Beta range from 0.1 to 0.9

# Double Exponential Smoothing function for FD
def des_FD(x, alpha_FD, beta_FD):
    df_all_FD = pd.DataFrame()
    df_all_FD['y'] = x
    df_all_FD['x'] = range(1, len(df_all_FD) + 1)
    df_all_FD.loc[len(df_all_FD), 'x'] = len(df_all_FD) + 1

    model_FD = ExponentialSmoothing(df_all_FD['y'], trend='add', seasonal=None)
    fitted_model_FD = model_FD.fit(smoothing_level=alpha_FD, smoothing_trend=beta_FD, optimized=False)
    
    return fitted_model_FD.fittedvalues.tolist()

# Function to find the best alpha & beta using Grid Search with RMSE for FD
def optimize_des_FD(series_FD):
    best_alpha_FD, best_beta_FD, best_rmse_FD = None, None, float("inf")

    for alpha_FD, beta_FD in product(alpha_values_FD, beta_values_FD):
        try:
            predictions_FD = des_FD(series_FD, alpha_FD, beta_FD)
            rmse_FD = np.sqrt(mean_squared_error(series_FD, predictions_FD[:len(series_FD)]))  # RMSE calculation

            if rmse_FD < best_rmse_FD:
                best_alpha_FD, best_beta_FD, best_rmse_FD = alpha_FD, beta_FD, rmse_FD

        except Exception as e:
            continue  # Skip if model fails for some values

    return best_alpha_FD, best_beta_FD

# Apply Grid Search Optimization for FD
df_all[['best_alpha_FD', 'best_beta_FD']] = df_all['clipped_d_FD'].apply(lambda x: pd.Series(optimize_des_FD(x)))
df_all['des_FD'] = df_all['clipped_d_FD'].apply(lambda x: des_FD(x, *optimize_des_FD(x)))
df_all['des_result_FD'] = df_all['des_FD'].apply(lambda x: x[-1:])  # Get last predicted value

display(df_all.tail())


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,pr2_FD,pr2_result_FD,pr3_FD,pr3_result_FD,ses_FD,ses_result_FD,best_alpha_FD,best_beta_FD,des_FD,des_result_FD
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,"[11.824004615566205, 10.88060614623838, 9.92836368774303, 8.96727724008015, 7.9973468032497435, 7.018572377251809, 6.030953962086348, 5.034491557753358, 4.029185164252841, 3.015034781584795, 1.992...",[-0.0804803014245028],"[11.755905898976167, 10.886796938655609, 9.971699234663891, 9.018867176890712, 8.036555155225766, 7.033017559558751, 6.016508779779365, 4.995283205777302, 3.9775952274422597, 2.9716992346639337, 1...",[0.10730706856622163],"[11.678078794301994, 11.678078794301992, 11.135615758860398, 10.22712315177208, 9.245424630354416, 8.249084926070884, 7.249816985214177, 6.249963397042836, 5.249992679408567, 4.249998535881714, 3....",[1.2499999882870536],0.1,0.1,"[11.88879085621341, 10.883171867895772, 9.91147518030074, 8.937833409662206, 7.962177481990904, 6.984465372266821, 6.004679819792479, 5.022826024367647, 4.038929348241621, 3.0530330462457815, 2.06...",[0.08400113747377091]
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,"[23.64800923113241, 21.76121229247676, 19.85672737548606, 17.9345544801603, 15.994693606499487, 14.037144754503618, 12.061907924172695, 10.068983115506716, 8.058370328505681, 6.03006956316959, 3.9...",[-0.1609606028490056],"[23.511811797952333, 21.773593877311217, 19.943398469327782, 18.037734353781424, 16.073110310451533, 14.066035119117503, 12.03301755955873, 9.990566411554603, 7.955190454884519, 5.943398469327867,...",[0.21461413713244326],"[23.35615758860399, 23.356157588603985, 22.271231517720796, 20.45424630354416, 18.49084926070883, 16.498169852141768, 14.499633970428354, 12.499926794085672, 10.499985358817135, 8.499997071763428,...",[2.499999976574107],0.1,0.1,"[23.77758171242682, 21.766343735791544, 19.82295036060148, 17.87566681932441, 15.924354963981807, 13.968930744533642, 12.009359639584957, 10.045652048735294, 8.077858696483242, 6.106066092491563, ...",[0.16800227494754183]
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,"[47.29601846226482, 43.52242458495352, 39.71345475097212, 35.8691089603206, 31.989387212998974, 28.074289509007237, 24.12381584834539, 20.137966231013433, 16.116740657011363, 12.06013912633918, 7....",[-0.3219212056980112],"[47.02362359590467, 43.547187754622435, 39.886796938655564, 36.07546870756285, 32.146220620903065, 28.132070238235006, 24.06603511911746, 19.981132823109206, 15.910380909769039, 11.886796938655735...",[0.4292282742648865],"[46.71231517720798, 46.71231517720797, 44.54246303544159, 40.90849260708832, 36.98169852141766, 32.996339704283535, 28.999267940856708, 24.999853588171344, 20.9999707176342

In [90]:
logging.info("BEGIN Metric Calculation for _FD")

# Calculate metrics including MdRAE, GMRAE, MASE, MAPE, and SMAPE
def metric_FD(x):
    period_length = len(x['clipped_d_FD'])
    df_all = pd.DataFrame()
    df_all['qty'] = x['clipped_d_FD'][:period_length]  # Ground truth values

    # Naive forecast (previous period's value)
    df_all['naive'] = df_all['qty'].shift(1)

    models = ['ma_FD', 'wma_forecast_FD', 'ewma_FD', 'lr_FD', 'pr2_FD', 'pr3_FD', 'ses_FD', 'des_FD']
    for model in models:
        df_all[model] = x[model][:period_length]

    # Compute MASE scaling factor (denominator)
    naive_diff = np.abs(df_all['qty'].diff()).dropna()
    naive_mae = naive_diff.mean() if not naive_diff.empty else np.nan

    result = []
    for model in models:
        y_true = df_all['qty'].dropna()
        y_pred = df_all[model].dropna()
        y_naive = df_all['naive'].dropna()

        # Standard error metrics
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        # Relative errors for MdRAE and GMRAE
        relative_errors = np.abs(y_true - y_pred) / np.abs(y_true - y_naive)
        relative_errors = relative_errors.replace([np.inf, -np.inf], np.nan).dropna()

        # Compute MdRAE and GMRAE
        if not relative_errors.empty:
            mdrae = np.median(relative_errors)
            gmrae = np.exp(np.mean(np.log(relative_errors)))
        else:
            mdrae, gmrae = np.nan, np.nan

        # Compute MASE
        mase = mae / naive_mae if naive_mae > 0 else np.nan

        # Compute MAPE (bounded between 0% - 100%)
        mape_values = np.abs((y_true - y_pred) / y_true)
        mape_values = mape_values.replace([np.inf, -np.inf], np.nan).dropna()
        mape = 100 * mape_values.mean() if not mape_values.empty else np.nan

        # Compute SMAPE (bounded between 0% - 100%)
        smape_values = np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-10)  # Avoid div by zero
        smape_values = smape_values.replace([np.inf, -np.inf], np.nan).dropna()
        smape = 100 * smape_values.mean() if not smape_values.empty else np.nan

        result.append({
            'model': model, 'RMSE': rmse, 'MAE': mae, 'R2': r2,
            'MdRAE': mdrae, 'GMRAE': gmrae, 'MASE': mase, 'MAPE': mape, 'SMAPE': smape
        })

    return result  # Returning the metrics list

# Apply the metric function
df_all['metrics_FD'] = df_all.apply(lambda x: metric_FD(x), axis=1)

def get_best_r2_FD(row):
    best_model = row['best_model']
    metrics_fd = row.get('metrics_FD', [])
    for m in metrics_fd:
        if m['model'] == best_model + '_FD':
            return m['R2']
    return np.nan
df_all['best_r2_FD'] = df_all.apply(get_best_r2_FD, axis=1)
df_all['r2_status_FD'] = np.where(df_all['best_r2_FD'] < 0.25, "R2 < 0.25", "Good")

display(df_all)

2025-07-30 15:59:45,880 - INFO - BEGIN Metric Calculation for _FD


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,pr3_result_FD,ses_FD,ses_result_FD,best_alpha_FD,best_beta_FD,des_FD,des_result_FD,metrics_FD,best_r2_FD,r2_status_FD
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,[0.10730706856622163],"[11.678078794301994, 11.678078794301992, 11.135615758860398, 10.22712315177208, 9.245424630354416, 8.249084926070884, 7.249816985214177, 6.249963397042836, 5.249992679408567, 4.249998535881714, 3....",[1.2499999882870536],0.1,0.1,"[11.88879085621341, 10.883171867895772, 9.91147518030074, 8.937833409662206, 7.962177481990904, 6.984465372266821, 6.004679819792479, 5.022826024367647, 4.038929348241621, 3.0530330462457815, 2.06...",[0.08400113747377091],"[{'model': 'ma_FD', 'RMSE': 1.7830342568337052, 'MAE': 1.6685686879392214, 'R2': 0.7266250258661726, 'MdRAE': 2.0, 'GMRAE': 1.8148706510636983, 'MASE': 1.718872460196265, 'MAPE': 48.77385233355213...",0.99982,Good
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,[0.21461413713244326],"[23.35615758860399, 23.356157588603985, 22.271231517720796, 20.45424630354416, 18.49084926070883, 16.498169852141768, 14.499633970428354, 12.499926794085672, 10.499985358817135, 8.499997071763428,...",[2.499999976574107],0.1,0.1,"[23.77758171242682, 21.766343735791544, 19.82295036060148, 17.87566681932441, 15.924354963981807, 13.968930744533642, 12.009359639584957, 10.045652048735294, 8.077858696483242, 6.106066092491563, ...",[0.16800227494754183],"[{'model': 'ma_FD', 'RMSE': 3.5660685136674104, 'MAE': 3.337137375878443, 'R2': 0.7266250258661726, 'MdRAE': 2.0, 'GMRAE': 1.8148706510636983, 'MASE': 1.718872460196265, 'MAPE': 48.773852333552135...",0.99982,Good
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,[0.4292282742648865],"[46.71231517720798, 46.71231517720797, 44.54246303544159, 40.90849260708832, 36.98169852141766, 32.996339704283535, 28.999267940856708, 24.999853588171344, 20.99997071763427, 16.999994143526855, 1...",[4.999999953148214],0.1,0.1,"[47.55516342485364, 43.53268747158309, 39.64590072120296, 35.75133363864882, 31.848709927963615, 27.937861489067284, 24.018719279169915, 20.091304097470587, 16.155717392966483, 12.212132184983126,...",[0.33600454989508366],"[{'model': 'ma_FD', 'RMSE': 7.132137027334821, 'MAE': 6.674274751756886, 'R2': 0.7266250258661726, 'MdRAE': 2.0, 'GMRAE': 1.8148706510636983, 'MASE': 1.718872460196265, 'MAPE': 48.773852333552135,...",0.99982,Good
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 101.42463035441595, 101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 1

In [ ]:
def apply_best_model_forecast(row):
    best_model = row['best_model']
    if best_model == 'ma':
        return row['ma_result_FD']
    elif best_model == 'wma':
        return row['wma_result']
    elif best_model == 'ewma':
        return row['ewma_result_FD']
    elif best_model == 'lr':
        return row['lr_result_FD'][-1] if isinstance(row['lr_result_FD'], list) else row['lr_result_FD']
    elif best_model == 'pr2':
        return row['pr2_result_FD'][-1] if isinstance(row['pr2_result_FD'], list) else row['pr2_result_FD']
    elif best_model == 'pr3':
        return row['pr3_result_FD'][-1] if isinstance(row['pr3_result_FD'], list) else row['pr3_result_FD']
    elif best_model == 'ses':
        return row['ses_result_FD'][-1] if isinstance(row['ses_result_FD'], list) else row['ses_result_FD']
    elif best_model == 'des':
        return row['des_result_FD'][-1] if isinstance(row['des_result_FD'], list) else row['des_result_FD']
    else:
        return np.nan
    
df_all['FD_forecast'] = df_all.apply(apply_best_model_forecast, axis=1)
# Define the number of months (from 12 to 1, excluding 0)
num_months = 13  # Total months (D-12 to D-0), but we exclude D-0

# Map best model to the correct forecast series (excluding pred_0_FD)
def extract_forecast_values(row, month_idx):
    best_model = row['best_model']
    forecast_column = f"{best_model}_FD"  # Example: 'ma_result_FD', 'wma_result_FD'
    
    if forecast_column in row and isinstance(row[forecast_column], list):
        if len(row[forecast_column]) >= (13 - month_idx):
            return row[forecast_column][12 - month_idx]  # Extract the correct past forecast
    return np.nan  # Return NaN if data is missing or not a list

# Create columns for pred_12_FD to pred_1_FD
for i in range(num_months - 1, 0, -1):  # From 12 to 1
    df_all[f'pred_{i}_FD'] = df_all.apply(lambda x: extract_forecast_values(x, i), axis=1)

# Ensure FD_forecast contains only numeric values
df_all['FD_final'] = np.maximum(0, df_all['FD_forecast'].round().astype(int))

# Get all columns except the last four we want to reorder
columns_to_keep = [col for col in df_all.columns if col not in ['best_model', 'metrics', 'FD_forecast', 'FD_final']]

# Define the new order with the last four columns at the end
column_order = columns_to_keep + ['best_model', 'metrics', 'FD_forecast', 'FD_final']

# Reorder DataFrame
df_all = df_all[column_order]

display(df_all)


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,pred_6_FD,pred_5_FD,pred_4_FD,pred_3_FD,pred_2_FD,pred_1_FD,best_model,metrics,FD_forecast,FD_final
0,ALL,81,10081 S,"[16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]",7.5,3.452053,12.678079,"[12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.678078794301994, 12.678078794301994, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0]","[12.678078794301994, 12.678078794301994, 12.452052529534663, 11.892692931433999, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]",...,6.016509,4.995283,3.977595,2.971699,1.985850,1.028301,pr3,"[{'model': 'ma', 'RMSE': 1.7830342568337052, 'MAE': 1.6685686879392214, 'R2': 0.7266250258661726, 'MdRAE': 2.0, 'GMRAE': 1.8148706510636983, 'MASE': 1.718872460196265, 'MAPE': 33.63097618903319, '...",0.107307,0
1,ALL,82,10082 A,"[32, 30, 28, 26, 24, 22, 20, 18, 16, 14, 12, 10, 8, 6, 4, 2]",15.0,6.904105,25.356158,"[25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 25.35615758860399, 25.35615758860399, 24.0, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]","[25.35615758860399, 25.35615758860399, 24.904105059069327, 23.785385862867997, 22.0, 20.0, 18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0]",...,12.033018,9.990566,7.955190,5.943398,3.971699,2.056602,pr3,"[{'model': 'ma', 'RMSE': 3.5660685136674104, 'MAE': 3.337137375878443, 'R2': 0.7266250258661726, 'MdRAE': 2.0, 'GMRAE': 1.8148706510636983, 'MASE': 1.718872460196265, 'MAPE': 33.63097618903319, 'S...",0.214614,0
2,ALL,82,10082 B,"[64, 60, 56, 52, 48, 44, 40, 36, 32, 28, 24, 20, 16, 12, 8, 4]",30.0,13.808210,50.712315,"[50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 50.71231517720798, 50.71231517720798, 48.0, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0, 8.0]","[50.71231517720798, 50.71231517720798, 49.80821011813865, 47.570771725735995, 44.0, 40.0, 36.0, 32.0, 28.0, 24.0, 20.0, 16.0, 12.0]",...,24.066035,19.981133,15.910381,11.886797,7.943398,4.113203,pr3,"[{'model': 'ma', 'RMSE': 7.132137027334821, 'MAE': 6.674274751756886, 'R2': 0.7266250258661726, 'MdRAE': 2.0, 'GMRAE': 1.8148706510636983, 'MASE': 1.718872460196265, 'MAPE': 33.63097618903319, 'SM...",0.429228,0
3,ALL,91,10081 S,"[128, 120, 112, 104, 96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8]",60.0,27.616420,101.424630,"[101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 101.42463035441595, 101.42463035441595, 96.0, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0, 16.0]","[101.42463035441595, 101.42463035441595, 99.6164202362773, 95.14154345147199, 88.0, 80.0, 72.0, 64.0, 56.0, 48.0, 40.0, 32.0, 24.0]",...,48.132070,39.962266,31.820762,23.773594,15.886797,8.226406,pr3,"[{'model': 'ma', 'RMSE': 14.264274054669642, 'MAE': 13.348549503513771, 'R2': 0.7266250258661726, 'MdRAE': 2.0, 'GMRAE': 1.8148706510636983, 'MASE': 1.718872460196265, 'MAPE': 33.63097618903319, '...",0.858457,0
4,ALL,92,10082 S,"[256, 240, 224, 208, 192, 176, 160, 144, 128, 112, 96, 80, 64, 48, 32, 16]",120.0,55.232840,202.849261,"[202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 202.8492607088319, 202.8492607088319, 192.0, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0, 32.0]","[202.8492607088319, 202.8492607088319, 199.2328404725546, 190.28308690294398, 176.0, 160.0, 144.0, 128.0, 112.0, 96.0, 80.0, 64.0, 48.0]",...,96.264140,79.924531,63.641524,47.547188,31.773594,16.452812,pr3,"[{'model': 'ma', 'RMSE': 28.528548109339283, 'MAE': 26.697099007027543, 'R2': 0.7266250258661726, 'MdRAE': 2.0, 'GMRAE': 1.8148706510636983, 'MASE': 1.718872460196265, 'MAPE': 33.63097618903319, '...",1.716913,1


In [92]:
logging.info("Forecast Calculation Completed")

2025-07-30 15:59:46,152 - INFO - Forecast Calculation Completed


In [93]:
logging.info("Begin Creating Excel For DataFrame")

# if output folder not exist, create it
if not os.path.exists("output"):
    os.makedirs("output")

# Create Excel File, filename with date
filename = "output/forecast_" + time.strftime("%Y-%m-%d") + ".xlsx"

# Save DataFrame to Excel
df_all.to_excel(filename, index=False)

# Get the file size in MB
file_size = os.path.getsize(filename) / (1024 * 1024)

logging.info(f"Excel File Created: {filename}, Size: {file_size:.2f} MB")



2025-07-30 15:59:46,168 - INFO - Begin Creating Excel For DataFrame
2025-07-30 15:59:46,219 - INFO - Excel File Created: output/forecast_2025-07-30.xlsx, Size: 0.02 MB
